In [3]:
from typing import Any
import re

def parse_heading(line: str) -> tuple[int, str] | None:
    match = re.compile(r"^(#{1,6})\s+(.+?)\s*$").match(line.strip())
    if not match:
        return None
    level = len(match.group(1))
    title = match.group(2).strip()
    return level, title

def split_into_sections(markdown: str) -> list[dict[str, Any]]:
    """Group markdown lines by heading, keeping the heading path as metadata."""
    sections: list[dict[str, Any]] = []
    heading_stack: list[tuple[int, str]] = []
    current_lines: list[str] = []
    current_path: list[str] = []

    def save_current_section() -> None:
        if current_lines:
            sections.append(
                {
                    "section_path": current_path.copy(),
                    "lines": current_lines.copy(),
                }
            )

    for raw_line in markdown.splitlines():
        line = raw_line.rstrip()
        heading = parse_heading(line)

        if heading is not None:
            save_current_section()
            current_lines = [line]

            level, title = heading
            while heading_stack and heading_stack[-1][0] >= level:
                heading_stack.pop()
            heading_stack.append((level, title))
            current_path = [heading_title for _level, heading_title in heading_stack]
            continue

        if not current_lines and not line.strip():
            continue
        current_lines.append(line)

    save_current_section()
    return sections


In [4]:
def parse_scalar(value: str) -> Any:
    value = value.strip()
    if len(value) >= 2 and value[0] == value[-1] == '"':
        value = value[1:-1].replace(r"\"", '"')
    if value.isdigit():
        return int(value)
    return value

def parse_front_matter(markdown: str) -> tuple[dict[str, Any], str]:
    """Read the simple YAML-like metadata written by convert_to_markdown.py."""
    lines = markdown.splitlines()
    if not lines or lines[0].strip() != "---":
        return {}, markdown

    metadata: dict[str, Any] = {}
    for index, line in enumerate(lines[1:], start=1):
        line = line.strip()
        if line == "---":
            body = "\n".join(lines[index + 1 :]).lstrip("\n")
            return metadata, body
        if ":" not in line:
            continue
        key, value = line.split(":", 1)
        metadata[key.strip()] = parse_scalar(value)

    return metadata, markdown

In [6]:
from pathlib import Path

page_paths = sorted(Path("/home/joaocrm/projects/norta-llm-hoi4-commander/data/interim/hoi4_wiki/pages").glob("*.md"))

for page_path in page_paths:
    print(page_path)

# metadata, body = parse_front_matter(page_path.read_text(encoding="utf-8"))

/home/joaocrm/projects/norta-llm-hoi4-commander/data/interim/hoi4_wiki/pages/00001-patch-1-16.md
/home/joaocrm/projects/norta-llm-hoi4-commander/data/interim/hoi4_wiki/pages/00002-patch-1-0-x.md
/home/joaocrm/projects/norta-llm-hoi4-commander/data/interim/hoi4_wiki/pages/00003-patch-1-14.md
/home/joaocrm/projects/norta-llm-hoi4-commander/data/interim/hoi4_wiki/pages/00004-patch-1-15.md
/home/joaocrm/projects/norta-llm-hoi4-commander/data/interim/hoi4_wiki/pages/00005-patch-1-13.md
/home/joaocrm/projects/norta-llm-hoi4-commander/data/interim/hoi4_wiki/pages/00006-patch-1-17.md
/home/joaocrm/projects/norta-llm-hoi4-commander/data/interim/hoi4_wiki/pages/00007-patch-1-12.md
/home/joaocrm/projects/norta-llm-hoi4-commander/data/interim/hoi4_wiki/pages/00008-patch-1-1.md
/home/joaocrm/projects/norta-llm-hoi4-commander/data/interim/hoi4_wiki/pages/00009-patch-1-18.md
/home/joaocrm/projects/norta-llm-hoi4-commander/data/interim/hoi4_wiki/pages/00010-patch-1-19.md
/home/joaocrm/projects/norta-l